In [ ]:
import kagglehub
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import os
import torch.nn as nn
from torchvision.models import efficientnet_v2_s

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# Load CIFAR-10 datasets
trainset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
testset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

# Create DataLoaders
trainloader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
testloader = DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
print(f"Number of training samples: {len(trainset)}")
print(f"Number of test samples: {len(testset)}")

In [ ]:
import os
import glob
from torch.utils.data import Dataset
from PIL import Image
import os
from torchvision.datasets import ImageFolder
from torchvision import transforms

class StandardDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        # 1. Auto-generate class dictionary (or define manually)
        # Sort ensures 0=Apple, 1=Banana consistently
        classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
        self.class_labels = {cls_name: i for i, cls_name in enumerate(classes)}

        self.image_paths = []
        self.labels = []

        # 2. Collect Paths
        for class_name, label in self.class_labels.items():
            paths = glob.glob(f"{root_dir}/{class_name}/*.*") # Matches .jpg, .png etc
            self.image_paths.extend(paths)
            self.labels.extend([label] * len(paths))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # 3. Load & Process
        image = Image.open(self.image_paths[idx]).convert("RGB")
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_dir = os.path.join(path, "Tire Textures", "training_data")
test_dir = os.path.join(path, "Tire Textures", "testing_data")

train_dataset = StandardDataset(root_dir=train_dir, transform=transform)
test_dataset = StandardDataset(root_dir=test_dir, transform=transform)


# OR


from torchvision.datasets import ImageFolder

# Automatically handles everything if folders are named correctly
train_dataset = ImageFolder(root=train_dir, transform=transform)
test_dataset  = ImageFolder(root=test_dir,  transform=transform)

In [ ]:
import pandas as pd

# Find and load the CSV file
csv_files = [f for f in os.listdir(fonts_path) if f.endswith('.csv')]
print(f"CSV files found: {csv_files}\n")

# Load the CSV
csv_path = os.path.join(fonts_path, csv_files[0])
df = pd.read_csv(csv_path)

print(f"CSV shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}\n")
print("First 10 rows:")
print(df.head(10))

print(f"\nUnique labels: {df['Font'].nunique()}")
print(f"Label distribution:\n{df['Font'].value_counts().head(10)}")

In [ ]:
class EMNIST(nn.Module):
    def __init__(self, num_classes=26):
        super(EMNIST, self).__init__()
        vgg16 = models.vgg16(weights=models.VGG16_Weights.DEFAULT)

        for param in vgg16.features.parameters():
            param.requires_grad = False

        self.features = vgg16.features

        self.classifier = nn.Sequential(
            nn.Linear(512 * 1 * 1, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, num_classes)
        )

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

model = EMNIST(num_classes=len(classes)).to(device)
print(model)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.classifier.parameters(), lr=0.001, momentum=0.9)

def validate_EMNIST(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

    epoch_loss = running_loss / total_samples
    epoch_accuracy = correct_predictions / total_samples
    return epoch_loss, epoch_accuracy